In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import optax
from flax import nnx

from qmodem import LSTM, CMAPSSAnalyst, create_dataloaders, split_cmapss
from qmodem.train import EarlyStopper, train_loop

BATCH_SIZE = 50
DROPUT_RATE = 0.3
HIDDEN_SIZE = 40
LR = 1e-2
N_EPOCHS = 100
NUM_SENSORS = 9
PATIENCE = 5
PRINT_EVERY = 10
SEED = 42
TIME_WINDOW_SIZE = 30

np.random.seed(SEED)

In [ ]:
analyst = CMAPSSAnalyst(
    relative_test_size=0.2
)  # the analyst splits the data already into `train_` and `test_`

train_df, test_df = analyst.train_df, analyst.test_df

# do another split for the validation set.
train_df, val_df = split_cmapss(train_df, relative_subset_size=0.3)

The analyst can compute the prognostic metrics of the different sensors, which will help retaining only those sensors which are most useful for prognostics.

In [ ]:
metrics_cmapss = analyst.compute_prognostic_metrics()
metrics_cmapss

Select the 9 sensors with the highest fitness.

In [ ]:
analyst.train_df.head()

In [ ]:
sensors_selected = metrics_cmapss.head(NUM_SENSORS)["sensor_name"].tolist()

# filter the dataframes to only include the selected sensors (exclude also the operati)
non_sensor_columns = [
    col
    for col in analyst.column_names
    if not (col.startswith("sensor") or col.startswith("op_setting"))
]
columns_selected = non_sensor_columns + sensors_selected
train_df = train_df[columns_selected]
val_df = val_df[columns_selected]
test_df = test_df[columns_selected]

In [ ]:
train_df.head()

Now, we can use the `CMAPSSDataSource` class to prepare the feature/label arrays for the train and test dataframes.
We use a standard scaler to bring the features to zero mean and variance = 1, in order to prevent the different scales of the features to condition the training.
We also define a time-window the histories, with a window length large enough to reveal degradation and small enough to prevent overfitting and have enough training samples.

In [ ]:
from sklearn.preprocessing import StandardScaler

from qmodem.data import CMAPSSDataSource

scaler = StandardScaler()

# Note, scaling happens at init time, so when the same scaler is passed to the test datasource, it will use the
# statistics computed on the training data to scale the test data.
ds_train = CMAPSSDataSource(
    train_df, train_or_test="train", scaler=scaler, window_size=TIME_WINDOW_SIZE
)
ds_val = CMAPSSDataSource(
    val_df, train_or_test="test", scaler=scaler, window_size=TIME_WINDOW_SIZE
)
ds_test = CMAPSSDataSource(
    test_df, train_or_test="test", scaler=scaler, window_size=TIME_WINDOW_SIZE
)

With the training and validation data sources, we define the dataloaders for batching during training.

In [ ]:
dl_train, dl_val = create_dataloaders(
    ds_train=ds_train,
    ds_val=ds_val,
    batch_size=BATCH_SIZE,
    seed_train=SEED,
    seed_val=SEED + 1,
    shuffle_train=False,
    shuffle_val=False,
)

Now we can define the data-processing model of our application. This has two LSTM layers, dropout after the LSTMs and a final dense layer. The dense layer reduces the feature dimension to a single output.

First, we will see just how the model processes a batch of inputs.

In [ ]:
model = LSTM(
    input_size=NUM_SENSORS,
    hidden_size=HIDDEN_SIZE,
    rngs=nnx.Rngs(0),
)

x0, y0 = next(iter(dl_train))
print(f"Input shape: {x0.shape}, target shape: {y0.shape}")
print(f"Model output shape: {model(x0, rngs=nnx.Rngs(0)).shape}")
print(y0)

At this point, we do the usual training setup (same as what we did for the battery RUL case).

In [ ]:
schedule = optax.cosine_decay_schedule(
    init_value=LR,
    decay_steps=N_EPOCHS * (len(ds_train) // BATCH_SIZE),
    alpha=0.1,
)
optimizer = nnx.Optimizer(model, optax.adam(schedule), wrt=nnx.Param)
early_stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4)


def mse_loss(model, batch):
    x, y = batch
    y_pred = model(x, rngs=nnx.Rngs(0))
    return ((y - y_pred) ** 2).mean()


@nnx.jit
def train_step(model, optimizer, batch):
    def loss_fn(model):
        return mse_loss(model, batch)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, batch):
    return mse_loss(model, batch)

And we're ready to train!

In [ ]:
best_val_loss, _ = train_loop(
    n_epochs=N_EPOCHS,
    dataloader_train=dl_train,
    dataloader_val=dl_val,
    train_batch_fn=lambda batch: train_step(model, optimizer, batch),
    eval_batch_fn=lambda batch: eval_step(model, batch),
    early_stopper=early_stopper,
    print_every=PRINT_EVERY,
    on_train_epoch_start=model.train,
    on_val_epoch_start=model.eval,
)

We can now pick the test cases from the split we did `CMAPSS/FD001/Train` and verify that the LSTM has learned the jet-engines RUL.

We will do this in two different ways:
1. Eval mode, where we will deactivate dropout and have no stochasticity on the prediction.
2. Train/MCD mode, where dropout will stay active and we will sample the trained model.

In [ ]:
test_df["unit_id"].unique()

In [ ]:
def plot_rul_eval(unit_id: int, ax: plt.Axes) -> None:
    X_u, y_u = ds_test.get_unit_arrays(unit_id=unit_id, window_size=TIME_WINDOW_SIZE)

    # Also get the time cycles for plotting, remembering to skip the first window to match
    # the times of the targets.
    t_u = test_df[test_df["unit_id"] == unit_id]["time_cycles"].values[
        TIME_WINDOW_SIZE - 1 :
    ]

    model.eval()
    y_pred_u = model(X_u, rngs=nnx.Rngs(0))

    ax.plot(t_u, y_u, label="True RUL")
    ax.plot(t_u, y_pred_u, label="Predicted RUL (LSTM)")
    ax.set_xlabel("Time Cycles")
    ax.set_ylabel("Remaining Useful Life (RUL)")
    ax.set_title(f"True vs Predicted RUL for Unit {unit_id}")
    ax.grid()
    ax.legend()

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 20))

for unit_id, ax in zip(test_df["unit_id"].unique(), axs.flatten()):
    plot_rul_eval(unit_id=unit_id, ax=ax)

fig.tight_layout()